# モデルファイル検査



In [ ]:

import json
import glob
import os
import hashlib

# safetensors index 検索
index_cands = glob.glob("/kaggle/input/*/model.safetensors.index.json")
print("safetensors index:", index_cands[:3])

if not index_cands:
    print("⚠ index.json なし。safetensors ファイル一覧:")
    for f in glob.glob("/kaggle/input/**/*.safetensors"):
        print(f"  {f}")
    print("⚠ モデルファイル検査はスキップ (dataset 未アタッチ)")
    raise SystemExit()

index_path = index_cands[0]
print(f"index: {index_path}")
idx = json.load(open(index_path))

metadata = idx.get("metadata", {})
weight_map = idx.get("weight_map", {})
print(f"total params: {metadata.get('total', 'unknown')}")

# shard ファイル検証
missing = []
bad_sha = []
for fname in weight_map.values():
    fpath = os.path.join(os.path.dirname(index_path), fname)
    if not os.path.exists(fpath):
        missing.append(fname)
        continue
    # SHA256 チェック (先頭 1MB のみ)
    h = hashlib.sha256()
    with open(fpath, "rb") as f:
        h.update(f.read(1024 * 1024))
    bad_sha.append((fname, h.hexdigest()[:12]))

print(f"\ntotal shards: {len(weight_map)}")
print(f"missing: {len(missing)}")
if missing:
    print("  missing:", missing[:5])
print(f"sha256 (sample): {len(bad_sha)}")
for name, sha in bad_sha[:5]:
    print(f"  {name}: {sha}")

# tokenizer 確認
tok_paths = glob.glob("/kaggle/input/**/tokenizer*.json", recursive=True) + \
            glob.glob("/kaggle/input/**/tokenizer*.model", recursive=True)
print(f"\ntokenizer files: {len(tok_paths)}")
for t in tok_paths[:3]:
    print(f"  {t} ({os.path.getsize(t) / 1024 / 1024:.1f} MB)")

# config.json
cfg_matches = glob.glob("/kaggle/input/*/config.json")
if not cfg_matches:
    raise SystemExit("config.json が見つからない")
cfg_path = cfg_matches[0]
cfg = json.load(open(cfg_path))
print(f"\nconfig.json:")
print(f"  model_type: {cfg.get('model_type')}")
print(f"  architectures: {cfg.get('architectures')}")
print(f"  num_attention_heads: {cfg.get('num_attention_heads')}")
print(f"  num_key_value_heads: {cfg.get('num_key_value_heads')}")
print(f"  hidden_size: {cfg.get('hidden_size')}")

print("\n=== モデルファイル検査完了 ===")
